In [ ]:
import pandas as pd
import numpy as np
import pickle

In [ ]:
def load_data(dataset, method):
    filename_merged_data = f"../data/intermediate/step3/{dataset}_{method}_merged_data.pkl"
    with open(filename_merged_data, "rb") as file:
        merged_data = pickle.load(file)
    return merged_data

In [ ]:
# Load data
nevo_openai_merged_data = load_data("nevo", "openai")
fdc_openai_merged_data = load_data("fdc", "openai")
kap_openai_merged_data = load_data("kap", "openai")

nevo_bow_merged_data = load_data("nevo", "bow")
fdc_bow_merged_data = load_data("fdc", "bow")
kap_bow_merged_data = load_data("kap", "bow")

nevo_tfidf_merged_data = load_data("nevo", "tfidf")
fdc_tfidf_merged_data = load_data("fdc", "tfidf")
kap_tfidf_merged_data = load_data("kap", "tfidf")

nevo_bert_merged_data = load_data("nevo", "bert")
fdc_bert_merged_data = load_data("fdc", "bert")
kap_bert_merged_data = load_data("kap", "bert")

In [ ]:
# NEVO
nr_no_match_nevo = nevo_openai_merged_data[pd.isna(nevo_openai_merged_data['best_match'])].shape[0]
# FDC
nr_no_match_fdc = fdc_openai_merged_data[pd.isna(fdc_openai_merged_data['best_match'])].shape[0]
# KAP
nr_no_match_kap = kap_openai_merged_data[pd.isna(kap_openai_merged_data['best_match'])].shape[0]

print("NEVO:", nr_no_match_nevo, "(", nr_no_match_nevo / nevo_openai_merged_data.shape[0] * 100, ")%")
print("FDC:", nr_no_match_fdc, "(", nr_no_match_fdc / fdc_openai_merged_data.shape[0] * 100, ")%")
print("KAP:", nr_no_match_kap, "(", nr_no_match_kap / kap_openai_merged_data.shape[0] * 100, ")%")

In [ ]:
# First, use the proposed foodon label if the label is 'proposed'
nevo_openai_merged_data['final_label'] = np.where(nevo_openai_merged_data['label'] == 'propose', 
                                                  nevo_openai_merged_data['proposed_foodon_label'].str.replace(r'[^\w\s]', '', regex=True), # Because of 'broader?' label
                                                  nevo_openai_merged_data['label'])
fdc_openai_merged_data['final_label'] = np.where(fdc_openai_merged_data['label'] == 'propose', 
                                                 fdc_openai_merged_data['proposed_foodon_label'].str.replace(r'[^\w\s]', '', regex=True), # Because of 'broader?' label
                                                 fdc_openai_merged_data['label'])
kap_openai_merged_data['final_label'] = np.where(kap_openai_merged_data['label'] == 'propose', 
                                                 kap_openai_merged_data['proposed_foodon_label'].str.replace(r'[^\w\s]', '', regex=True), # Because of 'broader?' label
                                                 kap_openai_merged_data['label'])

# Then, if 'final_label' is NaN, make it 'unknown'
nevo_openai_merged_data.fillna(value={'final_label': 'unknown'}, inplace=True)
fdc_openai_merged_data.fillna(value={'final_label': 'unknown'}, inplace=True)
kap_openai_merged_data.fillna(value={'final_label': 'unknown'}, inplace=True)

# Also, if 'best_match_plus_proposed' is NaN, make final_label unknown
nevo_openai_merged_data['final_label'] = np.where(nevo_openai_merged_data['best_match_plus_proposed'].isna(), "unknown", nevo_openai_merged_data['final_label'])
fdc_openai_merged_data['final_label'] = np.where(fdc_openai_merged_data['best_match_plus_proposed'].isna(), "unknown", fdc_openai_merged_data['final_label'])
kap_openai_merged_data['final_label'] = np.where(kap_openai_merged_data['best_match_plus_proposed'].isna(), "unknown", kap_openai_merged_data['final_label'])

In [ ]:
# Add final label to other datasets as well
nevo_bow_merged_data['final_label'] = nevo_openai_merged_data['final_label']
fdc_bow_merged_data['final_label'] = fdc_openai_merged_data['final_label']
kap_bow_merged_data['final_label'] = kap_openai_merged_data['final_label']

nevo_tfidf_merged_data['final_label'] = nevo_openai_merged_data['final_label']
fdc_tfidf_merged_data['final_label'] = fdc_openai_merged_data['final_label']
kap_tfidf_merged_data['final_label'] = kap_openai_merged_data['final_label']

nevo_bert_merged_data['final_label'] = nevo_openai_merged_data['final_label']
fdc_bert_merged_data['final_label'] = fdc_openai_merged_data['final_label']
kap_bert_merged_data['final_label'] = kap_openai_merged_data['final_label']

In [ ]:
def get_best_match(row):
    if pd.isna(row['pos_candidates']):
        return np.nan
    return row['similarity'][int(row['pos_candidates'])]

def create_summary_table(merged_data, column_name, method_name, dataset_name):
    # Make selection
    merged_data = merged_data[[column_name, 'best_match_plus_proposed', 'label', 'final_label', 'similarity', 'pos_candidates', 'position_match', 'hybrid_position_match']].copy()
    # Add similarity of best match
    merged_data['best_match_similarity'] = merged_data.apply(get_best_match, axis=1)
    # Add similarity of most similar item
    merged_data['highest_similarity'] = [merged_data['similarity'][i][0] for i in range(merged_data.shape[0])]
    # Add method name
    merged_data['method_name'] = method_name
    # Add dataset name
    merged_data['dataset'] = dataset_name
    # Difference between pos original and pos reranked
    merged_data['diff_position'] = merged_data['position_match'] - merged_data['pos_candidates']
    # Select and rename
    merged_data = merged_data[[column_name, 
                           'dataset',
                           'method_name',
                           'best_match_plus_proposed', 
                           'label',
                           'final_label', 
                           'similarity',
                           'best_match_similarity', 
                           'highest_similarity',
                           'diff_position',
                           'pos_candidates',
                           'position_match', 
                           'hybrid_position_match']].rename(
        columns={
            column_name: 'orig_name',
            'similarity': 'all_similarities',
            'pos_candidates': 'pos_original',
            'position_match': 'pos_reranking',
            'hybrid_position_match': 'pos_hybrid'
        }
    )
    # Add best method
    merged_data['best_method_orig_vs_rerank'] = np.where(
        merged_data['pos_original'] < merged_data['pos_reranking'], 'original',
        np.where(merged_data['pos_reranking'] < merged_data['pos_original'], 'reranked', 'equal')
    )

    merged_data['best_method_orig_vs_hybrid'] = np.where(
        merged_data['pos_original'] < merged_data['pos_hybrid'], 'original',
        np.where(merged_data['pos_hybrid'] < merged_data['pos_original'], 'hybrid', 'equal')
    )

    # Add length
    merged_data['length'] = [len(merged_data['orig_name'][i]) for i in range(merged_data.shape[0])]

    # Add ambiguity threshold
    merged_data['threshold'] = np.mean([np.mean(merged_data['all_similarities'][i]) for i in range(merged_data.shape[0])])

    # Add ranking category
    orig_conditions = [merged_data['pos_original'] == 0, merged_data['pos_original'] <= 2, merged_data['pos_original'] <= 4, merged_data['pos_original'] <= 14]
    reranked_conditions = [merged_data['pos_reranking'] == 0, merged_data['pos_reranking'] <= 2, merged_data['pos_reranking'] <= 4, merged_data['pos_reranking'] <= 14]
    hybrid_conditions = [merged_data['pos_hybrid'] == 0, merged_data['pos_hybrid'] <= 2, merged_data['pos_hybrid'] <= 4, merged_data['pos_hybrid'] <= 14]

    choices = ['top 1','top 3','top 5','top 15']

    merged_data['pos_original_cat'] = np.select(
        orig_conditions, choices, default='outside top 15'
    )
    merged_data['pos_reranking_cat'] = np.select(
        reranked_conditions, choices, default='outside top 15'
    )
    merged_data['pos_hybrid_cat'] = np.select(
        hybrid_conditions, choices, default='outside top 15'
    )

    return merged_data

In [ ]:
# Loop over all dataframes
# List of all dataframes and parameter values
summary_configs = [
    # OpenAI
    (nevo_openai_merged_data, 'nevo_name', 'openai', 'nevo'),
    (fdc_openai_merged_data, 'orig_name', 'openai', 'fdc'),
    (kap_openai_merged_data, 'orig_name', 'openai', 'kap'),
    # BoW
    (nevo_bow_merged_data, 'nevo_name', 'bow', 'nevo'),
    (fdc_bow_merged_data, 'orig_name', 'bow', 'fdc'),
    (kap_bow_merged_data, 'orig_name', 'bow', 'kap'),
    # Tf-Idf
    (nevo_tfidf_merged_data, 'nevo_name', 'tfidf', 'nevo'),
    (fdc_tfidf_merged_data, 'orig_name', 'tfidf', 'fdc'),
    (kap_tfidf_merged_data, 'orig_name', 'tfidf', 'kap'),
    # BERT
    (nevo_bert_merged_data, 'nevo_name', 'bert', 'nevo'),
    (fdc_bert_merged_data, 'orig_name', 'bert', 'fdc'),
    (kap_bert_merged_data, 'orig_name', 'bert', 'kap'),
]

# Add all results in a list
all_summaries = []

for data, column_name, method_name, dataset_name in summary_configs:
    summary = create_summary_table(data, column_name, method_name, dataset_name)
    all_summaries.append(summary)

# Concatenate all results in a final table
final_summary_table = pd.concat(all_summaries, ignore_index=True)

In [ ]:
tmp_openai = final_summary_table[final_summary_table['method_name'] == 'openai']
tmp_openai[tmp_openai['orig_name'].isin(['Black nightshade raw (Vegetables)', 'Fish, sunfish, pumpkin seed, cooked, dry heat (Finfish and Shellfish Products)'])]

In [ ]:
tmp_bert = final_summary_table[final_summary_table['method_name'] == 'bert']
tmp_bert[tmp_bert['orig_name'].isin(['Skyr skimmed plain (Milk and milk products)', 'Nuts, chestnuts, chinese, boiled and steamed (Nut and Seed Products)'])]

In [ ]:
df_annotated = final_summary_table[['dataset', 'orig_name', 'best_match_plus_proposed', 'final_label']].rename(columns={"best_match_plus_proposed": "best_match", "orig_name": "name", "final_label": "label"}).drop_duplicates().reset_index(drop=True)
df_annotated.to_csv("../data/results/df_annotated.csv")
df_annotated.to_excel("../data/results/df_annotated.xlsx")

In [ ]:
# In further evaluations, remove final_label equal to 'unknown'
final_summary_table = final_summary_table[final_summary_table['final_label']!='unknown']

## **Figures and tables for paper**

## Table 1

In [ ]:
# FoodOn
foodon_path = '../data/intermediate/df_foodon_processed.pkl'
df_foodon_processed = pd.read_pickle(foodon_path)
# Nevo
nevo_path = '../data/intermediate/df_nevo_processed.pkl'
df_nevo_processed = pd.read_pickle(nevo_path)
# FoodData Central
fooddatacentral_path = '../data/intermediate/df_fooddatacentral_processed.pkl'
df_fooddatacentral_processed = pd.read_pickle(fooddatacentral_path)
# KAP
kap_path = "../data/input/kap_embeddings.json"
df_kap = pd.read_json(kap_path, lines=True)

print(f"""**Nr of entities:**
FoodOn:\t\t  {df_foodon_processed.shape[0]},
Nevo:\t\t  {df_nevo_processed.shape[0]},
FoodData Central: {df_fooddatacentral_processed.shape[0]},
KAP:\t\t  {df_kap.shape[0]}
""")

## Table 2
*Table 2 is just an illustration, not based on actual data*

## Table 3

In [ ]:
# NEVO
table3_nevo_counts = nevo_openai_merged_data.value_counts('final_label')
table3_nevo_perc = nevo_openai_merged_data.value_counts('final_label') / 250 * 100
table3_nevo = table3_nevo_counts.astype(str) + " (" + table3_nevo_perc.round(1).astype(str) + "%)"
# FDC
table3_fdc_counts = fdc_openai_merged_data.value_counts('final_label')
table3_fdc_perc = fdc_openai_merged_data.value_counts('final_label') / 250 * 100
table3_fdc = table3_fdc_counts.astype(str) + " (" + table3_fdc_perc.round(1).astype(str) + "%)"
# KAP
table3_kap_counts = kap_openai_merged_data.value_counts('final_label')
table3_kap_perc = kap_openai_merged_data.value_counts('final_label') / 50 * 100
table3_kap = table3_kap_counts.astype(str) + " (" + table3_kap_perc.round(1).astype(str) + "%)"

# Complete table 3
df_table3 = pd.concat([table3_nevo, table3_fdc, table3_kap], axis=1)
df_table3.columns = ['NEVO', 'FoodData Central', 'KAP']
order = ['exact', 'close match', 'broader', 'narrow', 'related', 'unknown']
df_table3 = df_table3.loc[order]
df_table3

## Table 4

In [ ]:
from helpers.step2 import determine_results_traditional
from helpers.step3 import top_k_results

**OpenAI / text-embedding-ada-002**

In [ ]:
# NEVO
nevo_openai_original = determine_results_traditional(nevo_openai_merged_data, only_perc=True)
nevo_openai_target = top_k_results(nevo_openai_merged_data, position_column='position_match', only_perc=True)
nevo_openai_hybrid = top_k_results(nevo_openai_merged_data, position_column='hybrid_position_match', only_perc=True)
# KAP
kap_openai_original = determine_results_traditional(kap_openai_merged_data, only_perc=True)
kap_openai_target = top_k_results(kap_openai_merged_data, position_column='position_match', only_perc=True)
kap_openai_hybrid = top_k_results(kap_openai_merged_data, position_column='hybrid_position_match', only_perc=True)
# FDC
fdc_openai_original = determine_results_traditional(fdc_openai_merged_data, only_perc=True)
fdc_openai_target = top_k_results(fdc_openai_merged_data, position_column='position_match', only_perc=True)
fdc_openai_hybrid = top_k_results(fdc_openai_merged_data, position_column='hybrid_position_match', only_perc=True)

In [ ]:
# Combineer de data
top_k_list = [1,3,5,15]
table4_openai = []

for dataset, topks, orig, targ, hybr in [
    ('NEVO', top_k_list, nevo_openai_original, nevo_openai_target, nevo_openai_hybrid),
    ('KAP', top_k_list, kap_openai_original, kap_openai_target, kap_openai_hybrid),
    ('FoodData Central', top_k_list, fdc_openai_original, fdc_openai_target, fdc_openai_hybrid)
]:
    for i, k in enumerate(topks):
        table4_openai.append([dataset, k, orig[i], targ[i], hybr[i]])

# Maak de DataFrame
df_table4_openai = pd.DataFrame(table4_openai, columns=['Dataset', 'Top-k', 'Original', 'Target', 'Hybrid'])
df_table4_openai

**Bag-of-Words**

In [ ]:
# NEVO
nevo_bow_original = determine_results_traditional(nevo_bow_merged_data, only_perc=True)
nevo_bow_target = top_k_results(nevo_bow_merged_data, position_column='position_match', only_perc=True)
nevo_bow_hybrid = top_k_results(nevo_bow_merged_data, position_column='hybrid_position_match', only_perc=True)
# KAP
kap_bow_original = determine_results_traditional(kap_bow_merged_data, only_perc=True)
kap_bow_target = top_k_results(kap_bow_merged_data, position_column='position_match', only_perc=True)
kap_bow_hybrid = top_k_results(kap_bow_merged_data, position_column='hybrid_position_match', only_perc=True)
# FDC
fdc_bow_original = determine_results_traditional(fdc_bow_merged_data, only_perc=True)
fdc_bow_target = top_k_results(fdc_bow_merged_data, position_column='position_match', only_perc=True)
fdc_bow_hybrid = top_k_results(fdc_bow_merged_data, position_column='hybrid_position_match', only_perc=True)

In [ ]:
# Combineer de data
top_k_list = [1,3,5,15]
table4_bow = []

for dataset, topks, orig, targ, hybr in [
    ('NEVO', top_k_list, nevo_bow_original, nevo_bow_target, nevo_bow_hybrid),
    ('KAP', top_k_list, kap_bow_original, kap_bow_target, kap_bow_hybrid),
    ('FoodData Central', top_k_list, fdc_bow_original, fdc_bow_target, fdc_bow_hybrid)
]:
    for i, k in enumerate(topks):
        table4_bow.append([dataset, k, orig[i], targ[i], hybr[i]])

# Maak de DataFrame
df_table4_bow = pd.DataFrame(table4_bow, columns=['Dataset', 'Top-k', 'Original', 'Target', 'Hybrid'])
df_table4_bow

**tf-idf**

In [ ]:
# NEVO
nevo_tfidf_original = determine_results_traditional(nevo_tfidf_merged_data, only_perc=True)
nevo_tfidf_target = top_k_results(nevo_tfidf_merged_data, position_column='position_match', only_perc=True)
nevo_tfidf_hybrid = top_k_results(nevo_tfidf_merged_data, position_column='hybrid_position_match', only_perc=True)
# KAP
kap_tfidf_original = determine_results_traditional(kap_tfidf_merged_data, only_perc=True)
kap_tfidf_target = top_k_results(kap_tfidf_merged_data, position_column='position_match', only_perc=True)
kap_tfidf_hybrid = top_k_results(kap_tfidf_merged_data, position_column='hybrid_position_match', only_perc=True)
# FDC
fdc_tfidf_original = determine_results_traditional(fdc_tfidf_merged_data, only_perc=True)
fdc_tfidf_target = top_k_results(fdc_tfidf_merged_data, position_column='position_match', only_perc=True)
fdc_tfidf_hybrid = top_k_results(fdc_tfidf_merged_data, position_column='hybrid_position_match', only_perc=True)

In [ ]:
# Combineer de data
top_k_list = [1,3,5,15]
table4_tfidf = []

for dataset, topks, orig, targ, hybr in [
    ('NEVO', top_k_list, nevo_tfidf_original, nevo_tfidf_target, nevo_tfidf_hybrid),
    ('KAP', top_k_list, kap_tfidf_original, kap_tfidf_target, kap_tfidf_hybrid),
    ('FoodData Central', top_k_list, fdc_tfidf_original, fdc_tfidf_target, fdc_tfidf_hybrid)
]:
    for i, k in enumerate(topks):
        table4_tfidf.append([dataset, k, orig[i], targ[i], hybr[i]])

# Maak de DataFrame
df_table4_tfidf = pd.DataFrame(table4_tfidf, columns=['Dataset', 'Top-k', 'Original', 'Target', 'Hybrid'])
df_table4_tfidf

**SBERT**

In [ ]:
# NEVO
nevo_bert_original = determine_results_traditional(nevo_bert_merged_data, only_perc=True)
nevo_bert_target = top_k_results(nevo_bert_merged_data, position_column='position_match', only_perc=True)
nevo_bert_hybrid = top_k_results(nevo_bert_merged_data, position_column='hybrid_position_match', only_perc=True)
# KAP
kap_bert_original = determine_results_traditional(kap_bert_merged_data, only_perc=True)
kap_bert_target = top_k_results(kap_bert_merged_data, position_column='position_match', only_perc=True)
kap_bert_hybrid = top_k_results(kap_bert_merged_data, position_column='hybrid_position_match', only_perc=True)
# FDC
fdc_bert_original = determine_results_traditional(fdc_bert_merged_data, only_perc=True)
fdc_bert_target = top_k_results(fdc_bert_merged_data, position_column='position_match', only_perc=True)
fdc_bert_hybrid = top_k_results(fdc_bert_merged_data, position_column='hybrid_position_match', only_perc=True)

In [ ]:
# Combineer de data
top_k_list = [1,3,5,15]
table4_bert = []

for dataset, topks, orig, targ, hybr in [
    ('NEVO', top_k_list, nevo_bert_original, nevo_bert_target, nevo_bert_hybrid),
    ('KAP', top_k_list, kap_bert_original, kap_bert_target, kap_bert_hybrid),
    ('FoodData Central', top_k_list, fdc_bert_original, fdc_bert_target, fdc_bert_hybrid)
]:
    for i, k in enumerate(topks):
        table4_bert.append([dataset, k, orig[i], targ[i], hybr[i]])

# Maak de DataFrame
df_table4_bert = pd.DataFrame(table4_bert, columns=['Dataset', 'Top-k', 'Original', 'Target', 'Hybrid'])
df_table4_bert

## Table 5

In [ ]:
results_openai = final_summary_table[final_summary_table['method_name'] == 'openai']
results_bow = final_summary_table[final_summary_table['method_name'] == 'bow']
results_tfidf = final_summary_table[final_summary_table['method_name'] == 'tfidf']
results_sbert = final_summary_table[final_summary_table['method_name'] == 'bert']

In [ ]:
# Target ranking
all_counts_target = final_summary_table.value_counts(['best_method_orig_vs_rerank'], normalize=True) * 100
openai_counts_target = results_openai.value_counts(['best_method_orig_vs_rerank'], normalize=True) * 100
bow_counts_target = results_bow.value_counts(['best_method_orig_vs_rerank'], normalize=True) * 100
tfidf_counts_target = results_tfidf.value_counts(['best_method_orig_vs_rerank'], normalize=True) * 100
sbert_counts_target = results_sbert.value_counts(['best_method_orig_vs_rerank'], normalize=True) * 100

In [ ]:
combined_table_target = pd.concat(
    [
        all_counts_target.rename('All'),
        bow_counts_target.rename('BoW'),
        tfidf_counts_target.rename('TF-IDF'),
        sbert_counts_target.rename('SBERT'),
        openai_counts_target.rename('OpenAI')
    ],
    axis=1
).loc[['original', 'reranked', 'equal']]

combined_table_target

In [ ]:
# Hybrid
all_counts_hybrid = final_summary_table.value_counts(['best_method_orig_vs_hybrid'], normalize=True) * 100
openai_counts_hybrid = results_openai.value_counts(['best_method_orig_vs_hybrid'], normalize=True) * 100
bow_counts_hybrid = results_bow.value_counts(['best_method_orig_vs_hybrid'], normalize=True) * 100
tfidf_counts_hybrid = results_tfidf.value_counts(['best_method_orig_vs_hybrid'], normalize=True) * 100
sbert_counts_hybrid = results_sbert.value_counts(['best_method_orig_vs_hybrid'], normalize=True) * 100

In [ ]:
combined_table_hybrid = pd.concat(
    [
        all_counts_hybrid.rename('All'),
        bow_counts_hybrid.rename('BoW'),
        tfidf_counts_hybrid.rename('TF-IDF'),
        sbert_counts_hybrid.rename('SBERT'),
        openai_counts_hybrid.rename('OpenAI')
    ],
    axis=1
).loc[['original', 'hybrid', 'equal']]

combined_table_hybrid

## Figures

In [ ]:
from helpers.step4 import create_label_vs_ranking_table, plot_heatmap
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def plot_diffs_heatmap(results_openai, results_bow, results_tfidf, results_sbert,
                       hybrid = False, dataset_filter = "all"):
    if dataset_filter != "all":
        results_openai = results_openai[results_openai['dataset'] == dataset_filter]
        results_bow = results_bow[results_bow['dataset'] == dataset_filter]
        results_tfidf = results_tfidf[results_tfidf['dataset'] == dataset_filter]
        results_sbert = results_sbert[results_sbert['dataset'] == dataset_filter]
    # OpenAI
    ct_orig_openai = create_label_vs_ranking_table(results_openai, 'pos_original_cat')
    ct_reranking_openai = create_label_vs_ranking_table(results_openai, 'pos_reranking_cat')
    ct_hybrid_openai = create_label_vs_ranking_table(results_openai, 'pos_hybrid_cat')
    ct_diff_openai = ct_reranking_openai - ct_orig_openai
    ct_diff_hybrid_openai = ct_hybrid_openai - ct_orig_openai

    # BoW
    ct_orig_bow = create_label_vs_ranking_table(results_bow, 'pos_original_cat')
    ct_reranking_bow = create_label_vs_ranking_table(results_bow, 'pos_reranking_cat')
    ct_hybrid_bow = create_label_vs_ranking_table(results_bow, 'pos_hybrid_cat')
    ct_diff_bow = ct_reranking_bow - ct_orig_bow
    ct_diff_hybrid_bow = ct_hybrid_bow - ct_orig_bow

    # tf-idf
    ct_orig_tfidf = create_label_vs_ranking_table(results_tfidf, 'pos_original_cat')
    ct_reranking_tfidf = create_label_vs_ranking_table(results_tfidf, 'pos_reranking_cat')
    ct_hybrid_tfidf = create_label_vs_ranking_table(results_tfidf, 'pos_hybrid_cat')
    ct_diff_tfidf = ct_reranking_tfidf - ct_orig_tfidf
    ct_diff_hybrid_tfidf = ct_hybrid_tfidf - ct_orig_tfidf

    # S-BERT
    ct_orig_sbert = create_label_vs_ranking_table(results_sbert, 'pos_original_cat')
    ct_reranking_sbert = create_label_vs_ranking_table(results_sbert, 'pos_reranking_cat')
    ct_hybrid_sbert = create_label_vs_ranking_table(results_sbert, 'pos_hybrid_cat')
    ct_diff_sbert = ct_reranking_sbert - ct_orig_sbert
    ct_diff_hybrid_sbert = ct_hybrid_sbert - ct_orig_sbert

    # All
    fig, axs = plt.subplots(2, 2, figsize=(18, 12))

    if hybrid:
        dataframes = [ct_diff_hybrid_bow, ct_diff_hybrid_tfidf, ct_diff_hybrid_sbert, ct_diff_hybrid_openai]
        plot_path = f"../data/results/heatmaps_hybrid_target_{dataset_filter}.png"
    else:
        dataframes = [ct_diff_bow, ct_diff_tfidf, ct_diff_sbert, ct_diff_openai]
        plot_path = f"../data/results/heatmaps_absolute_target_{dataset_filter}.png"

    titles = ["Bag-of-Words", "tf-idf", "SBERT", "text-embedding-ada-002"]  # of geef elke heatmap een naam

    for ax, df, title in zip(axs.flat, dataframes, titles):
        plot_heatmap(df, percentage=False, ax=ax, title=title)

    plt.tight_layout()

    plt.savefig(plot_path, dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
plot_diffs_heatmap(results_openai, results_bow, results_tfidf, results_sbert,
                   hybrid = False, dataset_filter = "all")

In [ ]:
plot_diffs_heatmap(results_openai, results_bow, results_tfidf, results_sbert,
                   hybrid = True, dataset_filter = "all")

In [ ]:
plot_diffs_heatmap(results_openai, results_bow, results_tfidf, results_sbert,
                   hybrid = True, dataset_filter = "fdc")